# MenteUTP IA · Base de datos sintética y prueba de concepto

**Continuidad con Semana 09:** el estudiante escribe libremente en el prototipo (pantalla 03), recibe un resumen orientativo (04) y **solo si decide explorar opciones** se ejecuta la clasificación demostrativa (05). Los 12 wireframes originales se mantienen intactos.

Esta es **una base de 72 textos sintéticos NUEVA**, organizada según el diseño de la Semana 09. **No es una copia de los 72 contextos originales**, porque no se incluyó el notebook original. Nunca ingresar chats reales o datos identificables.

**Importante:** la clasificación no diagnostica ni recomienda un profesional específico. Los indicadores evalúan la consistencia con etiquetas inventadas para esta demostración académica.

In [ ]:
import io, csv
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report


## 1. Cargar el CSV sintético

Cuando aparezca el selector, sube **`MenteUTP_Entrenamiento_Sintetico_72.csv`**, incluido en el paquete.

In [ ]:
subidos = files.upload()
nombre = next((n for n in subidos if n.lower().endswith('.csv')), None)
if nombre is None:
    raise ValueError('Debes subir el archivo CSV incluido en el paquete')
registros = list(csv.DictReader(io.StringIO(subidos[nombre].decode('utf-8-sig'))))
assert len(registros) == 72, 'Se esperaban exactamente 72 ejemplos sintéticos'
X = np.array([r['contexto'] for r in registros])
y = np.array([r['opcion_atencion_afin'] for r in registros])
print('Número de ejemplos:', len(X))
print('Distribución:', {v:int((y==v).sum()) for v in sorted(set(y))})
print('Ejemplo ilustrativo:', registros[0])


## 2. Separación estratificada 75 % / 25 %

54 registros de entrenamiento y 18 registros de prueba; el vectorizador **TF-IDF se ajusta solo en entrenamiento** dentro de cada Pipeline.

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
assert (len(X_train),len(X_test)) == (54,18)
print('Entrenamiento:',len(X_train),'Prueba:',len(X_test))


## 3. Entrenamiento y comparación de los tres modelos

In [ ]:
algoritmos = {
    'Regresión Logística': LogisticRegression(max_iter=2000,random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100,random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42)
}
resultados={}; entrenados={}; predicciones={}
for nombre,estimador in algoritmos.items():
    pipeline=Pipeline([
        ('tfidf',TfidfVectorizer(strip_accents='unicode',ngram_range=(1,2))),
        ('clasificador',estimador)
    ])
    pipeline.fit(X_train,y_train)
    pred=pipeline.predict(X_test)
    entrenados[nombre],predicciones[nombre]=pipeline,pred
    resultados[nombre]={
        'Accuracy':accuracy_score(y_test,pred),
        'Precision macro':precision_score(y_test,pred,average='macro',zero_division=0),
        'Recall macro':recall_score(y_test,pred,average='macro',zero_division=0),
        'F1 macro':f1_score(y_test,pred,average='macro',zero_division=0)}
for n,met in resultados.items():
    print(n,':',', '.join(f'{k}={v:.1%}' for k,v in met.items()))
mejor=max(resultados,key=lambda n:(resultados[n]['F1 macro'],resultados[n]['Accuracy']))
print('Seleccionado para esta base nueva:',mejor)


## 4. Matriz de confusión y reporte

In [ ]:
etiquetas=['bienestar_universitario','estres_academico','situaciones_personales']
ConfusionMatrixDisplay.from_predictions(y_test,predicciones[mejor],labels=etiquetas,xticks_rotation=30,values_format='d')
plt.title('Matriz de confusión - nueva base sintética')
plt.tight_layout()
plt.show()
print(classification_report(y_test,predicciones[mejor],zero_division=0))


## 5. Sugerencia **solo con permiso** de explorar opciones

Esto simula el paso de la pantalla 04 a la 05. No almacena mensajes reales, no inicia reservas ni reemplaza la atención humana.

In [ ]:
def sugerir_opcion(contexto, consentimiento=False):
    if not consentimiento:
        return 'Sin predicción: el estudiante no solicitó explorar opciones.'
    if not isinstance(contexto,str) or not contexto.strip():
        return 'No hay contexto suficiente para esta demostración.'
    etiqueta=entrenados[mejor].predict([contexto])[0]
    return {'opcion_atencion_afin':etiqueta,'aviso':'Etiqueta académica ilustrativa, no diagnóstico ni decisión clínica.'}

texto_ficticio='Tengo parciales y entregas acumuladas y me cuesta organizar mis tareas'
print('Sin permiso:',sugerir_opcion(texto_ficticio,consentimiento=False))
print('Con permiso:',sugerir_opcion(texto_ficticio,consentimiento=True))
assert len(X_train)==54 and len(X_test)==18


## Interpretación y límites

Las puntuaciones se calcularon **solo con datos ficticios redactados para esta demostración**, por lo que no demuestran exactitud en estudiantes reales. Si el sistema se implementa, cualquier interacción debe contar con protección de datos, consentimiento explícito, revisión por especialistas y derivación humana. Los 12 wireframes originales no se sustituyen por este cuaderno.